# Labo LLM — Google Colab → GPU à la demande

Notebook orienté **Google Colab**. Il diagnostique le runtime, choisit automatiquement un quant selon le GPU détecté, installe llama.cpp, télécharge et lance le modèle, mesure réellement le débit, puis peut exposer l'API via un tunnel Cloudflare.

**Ordre :** 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9.

## Modèles disponibles (≥10B)

| Modèle | Params | VRAM Q4 | Abliterated | Spécialité |
|--------|--------|---------|-------------|------------|
| **Mistral-24B** ✨ | 24B | ~13.5 GB | ✅ | MMLU 81%, 128K ctx, équilibré |
| **Gemma-4-26B-MoE** | 26B MoE | ~13 GB | ✅ | Vision, 85 tok/s, 0.7% refusal |
| **Huihui-Qwen3.5-27B** | 27B | ~14 GB | ✅ | Meilleur reasoning abliterated |
| **DeepResearch-30B** | 30B MoE | ~14 GB | ❌ (base) | #1 benchmark redteam 2026 |

> Compatible GPU T4 15GB (Colab gratuit).

In [ ]:
# 1 — Diagnostic matériel Colab
import subprocess, shutil, os, re, json, urllib.request

def run(cmd, timeout=20):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except Exception as e:
        return "", str(e), 1

HW = {
    "env": "colab" if os.path.exists("/content") else "inconnu",
    "gpus": [],
    "cpu_count": os.cpu_count(),
    "ram_gb": None,
    "disk_free_gb": None,
    "cuda_version": None,
    "internet_ok": False,
}

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    out, err, rc = run([
        nvidia_smi,
        "--query-gpu=name,memory.total,compute_cap",
        "--format=csv,noheader"
    ])
    for line in out.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            name, mem, cc = parts[:3]
            mem_mib = int(re.sub(r"[^\d]", "", mem) or 0)
            HW["gpus"].append({
                "name": name,
                "vram_mib": mem_mib,
                "vram_gib": round(mem_mib / 1024, 2),
                "compute_capability": cc,
            })

    out, _, _ = run([nvidia_smi])
    m = re.search(r"CUDA Version:\s*([\d.]+)", out)
    if m:
        HW["cuda_version"] = m.group(1)

try:
    import psutil
    HW["ram_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
except Exception:
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    HW["ram_gb"] = round(int(line.split()[1]) / 1024**2, 1)
                    break
    except Exception:
        pass

try:
    disk = shutil.disk_usage("/content")
    HW["disk_free_gb"] = round(disk.free / 1024**3, 1)
except Exception:
    pass

try:
    urllib.request.urlopen("https://huggingface.co", timeout=8)
    HW["internet_ok"] = True
except Exception:
    pass

HW["gpu_count"] = len(HW["gpus"])
HW["total_vram_gib"] = round(sum(g["vram_gib"] for g in HW["gpus"]), 2)

print(json.dumps(HW, indent=2, ensure_ascii=False))

if not HW["internet_ok"]:
    print("\n❌ Internet sortant indisponible.")
elif HW["gpu_count"] == 0:
    print("\n⚠️ Aucun GPU détecté. Dans Colab : Runtime → Change runtime type → GPU.")
else:
    print(f"\n✅ {HW['gpu_count']} GPU(s), {HW['total_vram_gib']} GiB VRAM.")
    for g in HW["gpus"]:
        print(f"   • {g['name']} — {g['vram_gib']} GiB — CC {g['compute_capability']}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 — Choix du modèle + sélection automatique du quant
# ═══════════════════════════════════════════════════════════════

MODEL_CHOICE = "mistral-24b"

MODELS = {
    "mistral-24b": {
        "label": "Mistral-Small-3.2-24B abliterated (128K ctx, MMLU 81%)",
        "repo":  "mradermacher/Mistral-Small-3.2-24B-Instruct-2506-abliterated-GGUF",
        "repo_format": "mradermacher",
        "single_gpu_tiers": [("Q4_K_M", 13.5), ("Q3_K_M", 11.5), ("Q3_K_S", 10.8)],
        "dual_gpu_tiers":   [("Q5_K_M", 16.8), ("Q4_K_M", 13.5)],
        "cpu_tier": None,
        "ctx_default": 8192,
        "chat_template": "mistral",
    },
    "gemma4-26b-moe": {
        "label": "Gemma-4-26B-A4B abliterated (MoE, vision, ~4B actifs)",
        "repo":  "mradermacher/gemma-4-26B-A4B-it-abliterated-GGUF",
        "repo_format": "mradermacher",
        "single_gpu_tiers": [("Q4_K_M", 13.2), ("Q3_K_M", 11.4), ("Q3_K_S", 10.6)],
        "dual_gpu_tiers":   [("Q5_K_M", 16.5), ("Q4_K_M", 13.2)],
        "cpu_tier": None,
        "ctx_default": 8192,
        "chat_template": "gemma",
    },
    "huihui-qwen35-27b": {
        "label": "Huihui-Qwen3.5-27B abliterated (reasoning, Apache 2.0)",
        "repo":  "mradermacher/Huihui-Qwen3.5-27B-abliterated-GGUF",
        "repo_format": "mradermacher",
        "single_gpu_tiers": [("Q4_K_M", 14.0), ("Q3_K_M", 12.0), ("Q3_K_S", 11.2)],
        "dual_gpu_tiers":   [("Q5_K_M", 17.5), ("Q4_K_M", 14.0)],
        "cpu_tier": None,
        "ctx_default": 8192,
        "chat_template": "chatml",
    },
    "deepresearch-30b": {
        "label": "Tongyi-DeepResearch-30B-A3B (MoE, #1 redteam 2026)",
        "repo":  "bartowski/Alibaba-NLP_Tongyi-DeepResearch-30B-A3B-GGUF",
        "repo_format": "bartowski",
        "single_gpu_tiers": [("Q4_K_M", 14.0), ("Q3_K_M", 12.0), ("IQ3_M",  11.4)],
        "dual_gpu_tiers":   [("Q5_K_M", 17.5), ("Q4_K_M", 14.0)],
        "cpu_tier": None,
        "ctx_default": 8192,
        "chat_template": "chatml",
    },
}

if MODEL_CHOICE not in MODELS:
    raise ValueError("MODEL_CHOICE inconnu : " + MODEL_CHOICE)

def pick_quant(model_key, hw, safety_margin_gib=1.0):
    cfg = MODELS[model_key]
    n_gpu = hw["gpu_count"]
    if n_gpu == 0:
        if cfg["cpu_tier"]:
            return cfg["cpu_tier"][0], "cpu", cfg["cpu_tier"][1]
        print("⚠️  " + cfg["label"] + " : pas prévu en CPU pur.")
        return None, None, None
    if n_gpu >= 2:
        budget = hw["total_vram_gib"] - safety_margin_gib
        tiers  = cfg["dual_gpu_tiers"]
    else:
        budget = min(g["vram_gib"] for g in hw["gpus"]) - safety_margin_gib
        tiers  = cfg["single_gpu_tiers"]
    for tag, size in tiers:
        if size <= budget:
            return tag, "gpu", size
    tag, size = tiers[-1]
    print(f"⚠️  {tag} ({size} GiB) dépasse le budget ({budget:.1f} GiB) — on tente quand même.")
    return tag, "gpu", size

quant_tag, mode, quant_size = pick_quant(MODEL_CHOICE, HW)
cfg = MODELS[MODEL_CHOICE]

print("Modèle :", cfg["label"])
print("Dépôt  :", cfg["repo"])
print("Quant  :", quant_tag, "—", quant_size, "GiB")
print("Mode   :", mode)
print("Format :", cfg["repo_format"])

In [ ]:
# 3 — Installation de llama.cpp
import subprocess, os, shutil, sys

def sh(cmd, timeout=600):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode != 0 and r.stderr.strip():
        print(r.stderr[-2500:])
    return r

r = sh("curl -LsSf https://llama.app/install.sh | sh", timeout=180)

candidates = [
    shutil.which("llama"),
    os.path.expanduser("~/.local/bin/llama"),
    "/usr/local/bin/llama",
]

llama_bin = next((p for p in candidates if p and os.path.exists(p)), None)

if llama_bin:
    print("✅ llama trouvé :", llama_bin)
else:
    print("⚠️ Installeur officiel indisponible dans ce runtime.")
    print("→ Repli : compilation CUDA de llama.cpp")

    if not os.path.exists("llama.cpp"):
        sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp.git", timeout=300)

    cmake_args = "-DGGML_CUDA=ON"
    if HW["gpu_count"] > 0:
        cc = HW["gpus"][0].get("compute_capability")
        if cc:
            cmake_args += f' -DCMAKE_CUDA_ARCHITECTURES="{cc.replace(".", "")}"'

    sh(
        f"cmake -B llama.cpp/build -S llama.cpp "
        f"{cmake_args} -DCMAKE_BUILD_TYPE=Release",
        timeout=300
    )
    sh(
        "cmake --build llama.cpp/build -j$(nproc) "
        "--target llama-server llama-cli",
        timeout=1800
    )

    llama_bin = "llama.cpp/build/bin/llama-server"

LLAMA_SERVER_CMD = llama_bin if llama_bin and "llama-server" in os.path.basename(llama_bin) else None

if LLAMA_SERVER_CMD is None:
    if llama_bin:
        LLAMA_SERVER_CMD = f"{llama_bin} serve"

print("LLAMA_SERVER_CMD =", LLAMA_SERVER_CMD)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 4 — Téléchargement intelligent via HF (multi-format)
# ═══════════════════════════════════════════════════════════════
import os, re
from huggingface_hub import hf_hub_download, list_repo_files

os.makedirs("./models", exist_ok=True)
os.environ["HF_HUB_DISABLE_XET"] = "1"

def find_gguf_file(repo_id, quant_tag):
    try:
        files = list(list_repo_files(repo_id))
    except Exception as e:
        print("❌ Impossible de lister", repo_id, ":", str(e))
        return None

    gguf_files = [f for f in files if f.endswith(".gguf") and "-split-" not in f and ".part" not in f]

    tag_lower = quant_tag.lower()
    for f in gguf_files:
        if tag_lower in f.lower():
            return f

    if gguf_files:
        print("⚠️  Tag", quant_tag, "non trouvé — fichiers disponibles :")
        for f in gguf_files[:8]:
            print("   •", f)
        return None

    return None

def download_model(repo_id, quant_tag):
    filename = find_gguf_file(repo_id, quant_tag)
    if not filename:
        return None

    local_path = os.path.join("./models", os.path.basename(filename))

    if os.path.exists(local_path) and os.path.getsize(local_path) > 1_000_000:
        print("✅ Déjà téléchargé :", local_path)
        return local_path

    print("📥 Téléchargement :", filename, "depuis", repo_id)
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir="./models",
        )
        print("✅ Modèle :", path, " (", round(os.path.getsize(path)/1024**3, 1), "GiB)")
        return path
    except Exception as e:
        print("❌ Échec :", str(e))
        return None

MODEL_PATH = download_model(cfg["repo"], quant_tag)

if MODEL_PATH:
    print("\n🎯 Prêt pour le serveur :", MODEL_PATH)
else:
    print("\n⚠️  Téléchargement échoué — vérifie :", cfg["repo"])


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5 — Serveur llama-server + shim Ollama (port 11434)
# Shim = compatibilité Ollama native pour deepseek-tui, aichat, etc.
# ═══════════════════════════════════════════════════════════════
import subprocess, os, time, json, requests as _req

assert MODEL_PATH and os.path.exists(MODEL_PATH), "Cellule 4 non exécutée."

API_TOKEN   = "6baba07e670bb0f5e9d4635858d73a23e3c6d7d71b708b867b0965abd5eb6424"
LLAMA_PORT  = 8000
OLLAMA_PORT = 11434
N_PARALLEL  = 3
CONTEXT     = cfg.get("ctx_default", 8192)
LLAMA_BIN   = LLAMA_SERVER_CMD.split()[0]

# ── Arrêt propre ──
subprocess.run("pkill -f 'llama serve' 2>/dev/null || true", shell=True)
time.sleep(2)

# ── Lancement llama-server ──
llama_cmd = (
    f"{LLAMA_BIN} serve "
    f"--model {MODEL_PATH} "
    f"--host 0.0.0.0 --port {LLAMA_PORT} "
    f"-ngl 999 -c {CONTEXT} "
    f"--parallel {N_PARALLEL} "
    f"--jinja "
    f"--api-key {API_TOKEN} "
    f"> llama_server.log 2>&1 &"
)

os.system(llama_cmd)
print(f"⏳ llama-server démarré (port {LLAMA_PORT}, {N_PARALLEL} sessions)...")

for i in range(60):
    try:
        r = _req.get(
            f"http://127.0.0.1:{LLAMA_PORT}/health",
            headers={"Authorization": "Bearer " + API_TOKEN},
            timeout=3
        )
        if r.status_code == 200:
            print(f"✅ llama-server prêt — http://127.0.0.1:{LLAMA_PORT}")
            break
    except:
        pass
    time.sleep(5)
    if i % 6 == 0:
        print(f"  ... {i*5}s")
else:
    print("❌ Timeout llama-server")
    import sys; sys.exit(1)

# ── Shim Ollama ──
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "httpx"], capture_output=True)

from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
import uvicorn, httpx

OLLAMA_MODEL_ALIAS = MODEL_CHOICE
OPENAI_BASE = f"http://127.0.0.1:{LLAMA_PORT}/v1"
OPENAI_HEADERS = {"Authorization": "Bearer " + API_TOKEN}

ollama = FastAPI()

@ollama.get("/api/tags")
async def tags():
    return {"models": [{"name": OLLAMA_MODEL_ALIAS + ":latest", "model": OLLAMA_MODEL_ALIAS, "size": int(quant_size * 1024**3)}]}

@ollama.get("/api/version")
async def version():
    return {"version": "0.3.14"}

@ollama.post("/api/chat")
async def chat(request: Request):
    body = await request.json()
    stream = body.get("stream", False)
    messages = body.get("messages", [])
    options = body.get("options", {})

    openai_payload = {
        "model": OLLAMA_MODEL_ALIAS,
        "messages": messages,
        "stream": stream,
        "temperature": options.get("temperature", 0.7),
        "max_tokens": options.get("num_predict", 1024),
    }

    async with httpx.AsyncClient(timeout=180) as client:
        if stream:
            async def generate():
                async with client.stream("POST", OPENAI_BASE + "/chat/completions", json=openai_payload, headers=OPENAI_HEADERS) as resp:
                    async for line in resp.aiter_lines():
                        if line.startswith("data: ") and line != "data: [DONE]":
                            try:
                                chunk = json.loads(line[6:])
                                content = chunk["choices"][0].get("delta", {}).get("content", "")
                                if content:
                                    yield json.dumps({"model": OLLAMA_MODEL_ALIAS, "message": {"role": "assistant", "content": content}, "done": False}) + "\n"
                            except:
                                pass
                yield json.dumps({"model": OLLAMA_MODEL_ALIAS, "done": True, "message": {"role": "assistant", "content": ""}}) + "\n"
            return StreamingResponse(generate(), media_type="application/x-ndjson")
        else:
            r = await client.post(OPENAI_BASE + "/chat/completions", json=openai_payload, headers=OPENAI_HEADERS)
            data = r.json()
            content = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {})
            return {"model": OLLAMA_MODEL_ALIAS, "message": {"role": "assistant", "content": content}, "done": True, "eval_count": usage.get("completion_tokens", 0)}

import multiprocessing

def _run_shim():
    uvicorn.run(ollama, host="0.0.0.0", port=OLLAMA_PORT, log_level="warning")

shim_process = multiprocessing.Process(target=_run_shim, daemon=True)
shim_process.start()

for _ in range(20):
    try:
        if _req.get(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2).status_code == 200:
            print(f"✅ Shim Ollama prêt — http://127.0.0.1:{OLLAMA_PORT}")
            break
    except:
        pass
    time.sleep(1)

print("\n" + "="*60)
print("RÉCAP — Endpoints")
print("="*60)
print(f"  OpenAI-compatible → http://127.0.0.1:{LLAMA_PORT}/v1")
print(f"     Auth: Bearer {API_TOKEN[:16]}...")
print(f"  Ollama natif      → http://127.0.0.1:{OLLAMA_PORT}")
print(f"  Sessions //       : {N_PARALLEL}")
print("="*60)

PORT = LLAMA_PORT


In [ ]:
# 6 — Test réel + mesure du débit
import requests, time

payload = {
    "model": OLLAMA_MODEL_ALIAS,
    "messages": [{"role": "user", "content": "Réponds uniquement par : OK, le serveur fonctionne."}],
}

t0 = time.time()
resp = requests.post(f"http://127.0.0.1:{OLLAMA_PORT}/api/chat", json=payload, timeout=180)
dt = time.time() - t0

print("HTTP :", resp.status_code)
data = resp.json()

if "message" in data:
    text = data["message"]["content"]
    eval_count = data.get("eval_count", 0)
    print("Réponse :", text)
    print(f"Temps   : {dt:.2f}s")
    if eval_count:
        print(f"Débit   : {eval_count / dt:.2f} tok/s")
else:
    print("❌ Réponse inattendue :", data)


In [ ]:
# 7 — Tunnel Cloudflare temporaire
import subprocess, os, re, time

cloudflared = "./cloudflared"

if not os.path.exists(cloudflared):
    r = subprocess.run(
        "curl -Ls https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -o cloudflared && chmod +x cloudflared",
        shell=True, capture_output=True, text=True, timeout=180,
    )
    if r.returncode != 0:
        print(r.stderr[-2000:])

tunnel_log = "cloudflared.log"

subprocess.Popen(
    [cloudflared, "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT,
)

public_url = None

for _ in range(40):
    time.sleep(2)
    if os.path.exists(tunnel_log):
        txt = open(tunnel_log, encoding="utf-8", errors="ignore").read()
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
        if m:
            public_url = m.group(0)
            break

if public_url:
    print("🌐 URL publique temporaire :", public_url)
    print("API OpenAI :", public_url + "/v1/chat/completions")
    with open("cloudflare_url.txt", "w") as f:
        f.write(public_url)
else:
    print("❌ URL du tunnel introuvable.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 8 — Heartbeat C2 vers VPS (secrets Colab auto)
# ═══════════════════════════════════════════════════════════════
import json, hmac, hashlib, urllib.request, time, os
from google.colab import userdata

try:
    vps_secret = userdata.get("vps")
    hex_secret = userdata.get("hex")
except Exception as e:
    print(f"❌ Erreur secrets Colab : {e}")
    print("   Définis 'vps' (IP:PORT) et 'hex' (secret) dans Secrets")
    raise

if ":" in vps_secret:
    VPS_IP, VPS_PORT = vps_secret.rsplit(":", 1)
    VPS_PORT = int(VPS_PORT)
else:
    VPS_IP = vps_secret
    VPS_PORT = 8765

SECRET = hex_secret

print(f"🎯 VPS : {VPS_IP}:{VPS_PORT}")
print(f"🔑 Secret : {SECRET[:8]}...{SECRET[-8:]}")

def get_public_url():
    try:
        with open("cloudflare_url.txt", "r") as f:
            return f.read().strip()
    except:
        return None

def send_heartbeat(url_publique, api_key):
    if not url_publique:
        print("❌ Aucune URL publique")
        return False

    payload = json.dumps({"url": url_publique, "api_key": api_key, "timestamp": int(time.time())}).encode()
    sig = hmac.new(SECRET.encode(), payload, hashlib.sha256).hexdigest()

    req = urllib.request.Request(
        f"http://{VPS_IP}:{VPS_PORT}/heartbeat",
        data=payload,
        headers={"Content-Type": "application/json", "X-Signature": sig},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            result = resp.read().decode()
            print("✅ Heartbeat envoyé :", result)
            return True
    except urllib.error.HTTPError as e:
        print(f"❌ Erreur HTTP {e.code} : {e.read().decode()}")
        return False
    except Exception as e:
        print(f"❌ Erreur : {e}")
        return False

print("\n🔄 Envoi du heartbeat au VPS...")

URL_PUBLIQUE = get_public_url()

if not URL_PUBLIQUE:
    print("⚠️ Aucune URL publique — exécute cellule 7 d'abord")
else:
    print(f"📡 URL : {URL_PUBLIQUE}")
    send_heartbeat(URL_PUBLIQUE, API_TOKEN)
    print("\n" + "="*50)
    print("📝 Vérif VPS :")
    print(f"   curl http://{VPS_IP}:{VPS_PORT}/health")
    print("="*50)


## Notes

- Pour changer de modèle : modifie `MODEL_CHOICE` en cellule 2 puis relance 2 → 8.
- Pour augmenter le contexte : modifie `CONTEXT` en cellule 5 et relance.
- Le shim Ollama tourne en multiprocessing pour permettre pkill propre.
- Le tunnel Cloudflare est temporaire : l'URL change à chaque session.
- **Tous les modèles ≥10B params** (minimum T4 15GB).